# Sesión 11 - Lab 1: Monitoreo, infraestructura y layout de datos

Este notebook corre entero sobre el compute **serverless** habitual del curso: ninguno de sus pasos necesita el Spark UI, ni siquiera los de layout de datos que cierran el notebook. El único punto de la sesión que sí lo necesita es el Lab Reto, para su propio diagnóstico; por eso el cluster clásico que creás acá en el Lab 1E queda disponible para cuando llegues ahí, aunque este notebook nunca se conecte a él.

A diferencia de los demás laboratorios del curso, acá no se ingesta ningún archivo nuevo. Se reutiliza el historial real de los jobs que ya desplegaste en las Sesiones 08, 09 y 10 para leer cómo se monitorea y se diagnostica un job, se crea puntualmente un cluster clásico para practicar dos fallos de infraestructura (arranque de compute y conflicto de librerías) que no dejan rastro en un notebook, y se cierra con tres técnicas de layout de datos (repartición, Liquid Clustering, predictive optimization) sobre un dataset sintético nuevo.

## Verificación del entorno

In [0]:
display(spark.sql("SHOW SCHEMAS IN dbassociate"))
print("Si ves bronze/silver/gold/default en la lista de arriba, el catalog está accesible desde este compute.")

## Lab 1A — Identificar el job a analizar

Elegí uno de los jobs que ya desplegaste en una sesión anterior (por ejemplo, `sesion08_pipeline_pedidos` de la Sesión 08, o el job que corre el pipeline de la Sesión 09/10). Abrí **Workflows → Jobs & Pipelines**, hacé clic en ese job, y copiá el `job_id` de la URL (el número que aparece después de `/jobs/`).

Pegalo en el widget de la celda de abajo antes de seguir.

In [0]:
dbutils.widgets.text("job_id", "")
dbutils.widgets.text("job_nombre_referencial", "sesion08_pipeline_pedidos")

job_id = dbutils.widgets.get("job_id")
job_nombre_referencial = dbutils.widgets.get("job_nombre_referencial")

if job_id == "":
    print("Falta completar el widget job_id con el ID real de un job de tu workspace.")
else:
    print(f"Analizando job_id={job_id} (referencia: {job_nombre_referencial})")

## Lab 1B — Línea base: duración, percentiles y tasa de falla

Los registros de `system.lakeflow.job_run_timeline`/`job_task_run_timeline` tardan **hasta una hora** en aparecer. Si acabás de correr el job y la consulta de abajo devuelve cero filas, no es un error: todavía no llegó el registro. Probá con un job que ya haya corrido en sesiones anteriores.

Para acceder a estos schemas de sistema hace falta ser metastore admin y account admin, o tener `USE` y `SELECT` sobre `system.lakeflow`.

In [0]:
# usamos tablas
# Vemos los detablas de el Job con id {job_id}
df_linea_base = spark.sql(f"""
WITH job_run_duration AS (
  SELECT
    workspace_id,
    job_id,
    run_id,
    CAST(SUM(period_end_time - period_start_time) AS LONG) AS duration_seconds,
    FIRST(result_state, true) AS result_state
  FROM system.lakeflow.job_run_timeline
  WHERE job_id = '{job_id}'
    AND period_start_time > CURRENT_TIMESTAMP() - INTERVAL 30 DAYS
  GROUP BY ALL
)
SELECT
  job_id,
  COUNT(*) AS total_runs,
  ROUND(AVG(duration_seconds), 1) AS avg_duration_seconds,
  PERCENTILE(duration_seconds, 0.5) AS p50_duration_seconds,
  PERCENTILE(duration_seconds, 0.9) AS p90_duration_seconds,
  PERCENTILE(duration_seconds, 0.95) AS p95_duration_seconds,
  ROUND(100.0 * SUM(CASE WHEN result_state = 'FAILED' THEN 1 ELSE 0 END) / COUNT(*), 1) AS tasa_falla_pct
FROM job_run_duration
GROUP BY job_id
""")

display(df_linea_base)

Esta es la línea base contra la que se compara cualquier corrida sospechosa. Un run puntual que tarde bastante más que `p90_duration_seconds`, o una racha de `tasa_falla_pct` que suba de golpe, son las dos señales concretas que justifican abrir el diagnóstico.

## Lab 1C — Forzar una falla y leer el DAG en vivo

El job `sesion08_pipeline_pedidos` (Sesión 08) ya trae un interruptor de falla forzada en la tarea `cuarentena_registros`, pensado en su momento para practicar Repair Run. Lo reutilizamos acá para leer el DAG con otra pregunta en mente: distinguir una tarea que falla de una que solo queda bloqueada como consecuencia.

1. Abrí ese job en **Workflows** y hacé clic en **Run now** con la configuración normal (todas las tareas con `forzar_fallo=false`). Debería terminar todo en verde.
2. Editá la tarea `cuarentena_registros`: en **Parameters**, cambiá `forzar_fallo` a `true`. Lanzá **Run now** de nuevo.
3. Mirá el DAG del nuevo run. Deberías ver tres estados distintos, no dos:
   - `ingesta_bronze` y `validar_calidad`: **Succeeded** (verde), no tienen nada que ver con la falla.
   - `transformar_silver`: **Skipped** (gris) porque la decisión de branching de `validar_calidad` no lo eligió, no por la falla.
   - `cuarentena_registros`: **Failed** (rojo), acá está la causa real.
   - `agregacion_gold`: bloqueada como consecuencia de que ninguna de sus dos dependencias tuvo éxito (la condición `At least one succeeded` no se cumple).
4. Volvé a poner `forzar_fallo=false` en `cuarentena_registros` para dejar el job como estaba.

El punto de esta lectura: dos tareas terminaron sin éxito (`transformar_silver` y `agregacion_gold`), pero solo una de las dos falló de verdad. Confundir las tres causas (decisión de branching, falla real, bloqueo por dependencia) es el error más común al leer un DAG apurado.

## Lab 1D — Taxonomía de códigos de terminación

`termination_code` (no `result_state`) es la columna que explica el motivo puntual de una tarea o un run que no terminó en éxito simple.

In [0]:
df_terminacion = spark.sql(f"""
SELECT
  termination_code,
  result_state,
  COUNT(*) AS cantidad
FROM system.lakeflow.job_task_run_timeline
WHERE job_id = '{job_id}'
  AND period_start_time > CURRENT_TIMESTAMP() - INTERVAL 30 DAYS
GROUP BY termination_code, result_state
ORDER BY cantidad DESC
""")

display(df_terminacion)

Familias más comunes de `termination_code` (catálogo completo en `material_apoyo_diagnostico_infraestructura.md`):

| Familia | Códigos típicos |
|---|---|
| Infraestructura de compute | `CLUSTER_ERROR`, `INVALID_CLUSTER_REQUEST`, `CLOUD_FAILURE` |
| Librerías y configuración | `LIBRARY_INSTALLATION_ERROR`, `INVALID_RUN_CONFIGURATION` |
| Ejecución y permisos | `RUN_EXECUTION_ERROR`, `DRIVER_ERROR`, `UNAUTHORIZED_ERROR` |
| Terminación normal | `SUCCESS`, `CANCELLED`, `SKIPPED` |

Este puente es el que conecta el bloque de monitoreo con el de infraestructura de los Labs 1E y 1F: cada fallo que vas a provocar a mano ahí abajo termina cayendo en una de estas mismas familias.

## Lab 1E — Fallo de arranque de cluster simulado

**Este cluster se reutiliza en el Lab 1F y queda disponible para el Lab Reto**: no hace falta crear uno nuevo ahí.

1. **Compute → Create compute.** Nombre: `sesion11_cluster_diagnostico`. Access mode: **Dedicated (single user)**. Databricks Runtime: **15.4 LTS o superior**. Tamaño: un único worker de tamaño mediano alcanza para este laboratorio.
2. Antes de crear, abrí **Advanced options → Init Scripts** y agregá un script apuntando a una ruta que a propósito no existe: `/Volumes/dbassociate/default/vol_landing/sesion_11/init_no_existe.sh`.
3. **Create compute.** El cluster va a intentar arrancar y no va a llegar a estado *Running*.
4. Mientras arranca (o después de que falle), abrí la pestaña **Event log** del cluster. Vas a encontrar un evento que marca la falla en la fase de inicialización, señalando que el script configurado no se pudo ejecutar.
5. Si esta ejecución estuviera asociada a un Job en vez de a un cluster interactivo, el `termination_code` que verías en las consultas del Lab 1D sería `CLUSTER_ERROR`: un fallo de arranque de compute no deja logs de aplicación, la evidencia está en el registro de eventos del compute, no en ningún notebook.
6. Editá el cluster, quitá el init script de la ruta inexistente, y arrancalo de nuevo. Ahora sí debería llegar a *Running*. Dejalo así: lo volvés a usar en el próximo paso.

## Lab 1F — Conflicto de librerías

Sobre el mismo cluster que acabás de dejar funcionando:

1. Abrí la pestaña **Libraries** del cluster y hacé clic en **Install New**.
2. Elegí **PyPI** y pedí una versión que no existe, por ejemplo `pandas==99.99.99`.
3. **Install.** La instalación va a terminar en estado **Failed**, con un mensaje de resolución de dependencias que no encuentra esa versión. Es determinista: siempre va a fallar, sin depender del Databricks Runtime que estés usando.

En la práctica, el conflicto más frecuente no es una versión inexistente sino una que sí existe pero es incompatible con lo que ya trae el runtime del cluster (por ejemplo, una versión de `numpy` o `pyarrow` más vieja que la que necesitan las librerías internas). Ese tipo de incompatibilidad concreta cambia de una versión de runtime a otra: el catálogo de `material_apoyo_diagnostico_infraestructura.md` trae más detalle, pero antes de mostrar un ejemplo así en vivo conviene verificarlo contra las notas de la versión del runtime que vayas a usar ese día.

4. Quitá la librería fallida antes de seguir, para no dejar el cluster en un estado confuso.

## Lab 1G — Firma de un out-of-memory (sin forzarlo en vivo)

Provocar un out-of-memory real reinicia el cluster y corta cualquier notebook que esté corriendo ahí, así que este paso queda descriptivo: reconocer la firma, no reproducirla en clase.

**Driver:** deja de responder o se reinicia solo, sin que el código del notebook muestre una excepción propia. En el registro de eventos del cluster se ve actividad sostenida de recolección de basura justo antes del reinicio. Causa típica: una operación que trae al driver más datos de los que debería (un `collect()` o un `toPandas()` sobre algo que no cabe en memoria).

**Executor:** casi nunca aparece de golpe. Primero se ve degradación de rendimiento sostenida en las métricas del stage (el mecanismo exacto está en `material_apoyo_spill.html`), y recién después la tarea falla con un error de memoria. Por eso conviene tratar esa degradación como una alerta temprana, no como un detalle menor.

## Lab 1H — Generar un dataset sintético para practicar layout de datos

Sin archivo estático: cada corrida de esta celda genera valores distintos. La columna `tienda_id` queda a propósito desbalanceada (una tienda concentra buena parte de las filas), porque el Lab Reto reutiliza este mismo patrón de generación para su propio dataset.

In [0]:
from pyspark.sql import functions as F

# creamos 6 millones de datos, esto para aplicar el repartition y coalesce.
N_FILAS = 6_000_000
TIENDA_DOMINANTE = 1
N_TIENDAS = 50

df_hechos = (
    spark.range(N_FILAS)
    .withColumnRenamed("id", "venta_id")
    .withColumn(
        "tienda_id",
        F.when(F.rand(seed=42) < 0.75, F.lit(TIENDA_DOMINANTE))
         .otherwise((F.floor(F.rand(seed=7) * (N_TIENDAS - 1)) + 2).cast("int")),
    )
    .withColumn("producto_id", (F.floor(F.rand(seed=11) * 500) + 1).cast("int"))
    .withColumn("monto", F.round(F.rand(seed=13) * 480 + 20, 2))
)
df_hechos.write.mode("overwrite").saveAsTable("dbassociate.default.sesion11_ventas_hechos")

print(f"sesion11_ventas_hechos: {spark.table('dbassociate.default.sesion11_ventas_hechos').count()} filas")

## Lab 1I — `repartition()`: cuándo ayuda y cuándo no

`repartition()` resuelve un problema puntual: particiones de **entrada** mal dimensionadas antes de una transformación ancha, por ejemplo cuando el archivo de origen llegó empaquetado en pocos archivos grandes. No es una técnica para balancear una llave puntual entre las particiones de salida de un shuffle; para eso están `material_apoyo_skew.html` y `material_apoyo_salting.html`.

In [0]:
# repartition() reorganiza los datos en diferentes particiones para que puedan ejecutarse en un worker para ganar procesamiento en paralelo.
# coalesce() disminuye las particiones de trabajo.

**Para explicarlo en clase:** `repartition()` es reorganizar las cajas antes de empezar a trabajar, no mientras se está sumando una sucursal puntual. Si el archivo de origen quedó mal empaquetado y terminaste con 4 cajas gigantes en vez de 64 cajas parejas, no importa cuántas veces reordenes esas 4 cajas al final: eso no cambia cuánto trabajo hay adentro de cada una. Lo que `repartition()` sí resuelve es que, desde el arranque, el trabajo ya esté parejo entre varias personas, para que la transformación que sigue no dependa de 4 personas cargando cajas enormes mientras el resto del equipo espera sin nada que hacer.

**Desglose del código, paso a paso:**

1. `df_pocas_particiones = df_hechos.coalesce(4)` junta los datos de `df_hechos` en apenas 4 particiones, sin redistribuirlos por ninguna llave (a diferencia de un `groupBy`, `coalesce()` solo pega particiones existentes entre sí, sin disparar shuffle). El resultado es el mismo contenido, pero repartido en muchas menos porciones, cada una más grande.

2. `df_pocas_particiones.groupBy(F.spark_partition_id().alias("particion")).count()` es una forma de ver, sin usar la API de RDD, cuántas filas cayeron en cada partición física: `spark_partition_id()` devuelve, para cada fila, el número de partición donde vive en ese momento. Agrupar por ese número y contar te da la foto de cómo quedó repartido el trabajo.

3. En la segunda celda, `df_pocas_particiones.groupBy("producto_id").agg(...).count()` corre una agregación por producto sobre ese DataFrame de solo 4 particiones grandes, y `time.time()` mide cuánto tarda. El `.count()` al final de la cadena es lo que fuerza la ejecución real: sin él, la agregación quedaría sin disparar.

4. `df_repartido = df_hechos.repartition(64)` vuelve a redistribuir `df_hechos` (el original, no el de 4 particiones) en 64 particiones parejas, esta vez sí con un shuffle de por medio. La misma agregación por `producto_id` corre después sobre este DataFrame, y se mide el tiempo de la misma forma para comparar los dos resultados de manera justa.

In [0]:
# antes de realizar una acción, como un .groupBy, lo dividimos en 4 particiones y luego ejecucuta la acción
df_pocas_particiones = df_hechos.coalesce(4)

display(
    # .spark_partition_id() retorna el número o id de la partición, es decir, si tenemos 4, sus id irán del 0 a 3
    df_pocas_particiones.groupBy(F.spark_partition_id().alias("particion")).count().orderBy("particion")
)
# Cuatro particiones enormes en vez de las que traía la tabla original: un caso de layout de entrada mal
# dimensionado, no de una llave desbalanceada.

In [0]:
import time
# Vemos la diferencia de tiempos entre 4 (1.1s) y 64 (1s) particiones

# con el coalesce de 4 particiones
t0 = time.time()
grupos_lento = df_pocas_particiones.groupBy("producto_id").agg(F.sum("monto").alias("total")).count()
t1 = time.time()
print(f"Con 4 particiones de entrada ({grupos_lento} grupos de producto_id): {t1 - t0:.1f} s")

# repartimos los datos en 64 particiones, de 4 a 64
df_repartido = df_hechos.repartition(64)
t0 = time.time()
grupos_rapido = df_repartido.groupBy("producto_id").agg(F.sum("monto").alias("total")).count()
t1 = time.time()
print(f"Después de repartition(64) ({grupos_rapido} grupos de producto_id): {t1 - t0:.1f} s")

## Lab 1J — Liquid Clustering

`CLUSTER BY` reemplaza al particionamiento tradicional y al Z-Ordering: organiza los archivos de la tabla por la llave elegida, y esa llave se puede cambiar más adelante sin reescribir los datos existentes. Admite hasta cuatro llaves de clustering.

**Para explicarlo en clase:** pensá en un archivador de oficina. Si guardás los papeles ordenados por sucursal desde el primer día, cuando alguien pide "dame solo los papeles de la sucursal 37" no hace falta revisar el archivador entero: vas directo al cajón de esa sucursal. Liquid Clustering hace exactamente eso con los archivos de la tabla: los ordena por la columna que más se usa para filtrar. A diferencia de un archivador físico, más adelante podés decirle "che, ahora ordenalo por otra columna" sin tener que sacar y volver a guardar todos los papeles a mano.

**Desglose del código, paso a paso:**

1. `CREATE OR REPLACE TABLE ... CLUSTER BY (tienda_id) AS SELECT * FROM ...` crea una tabla nueva con el mismo contenido que `sesion11_ventas_hechos`, pero declarando desde el vamos que se va a organizar por `tienda_id`. `CLUSTER BY` reemplaza acá al `PARTITIONED BY`/`ZORDER BY` que verías en una tabla tradicional.

2. `OPTIMIZE ... FULL` es el paso que efectivamente reorganiza los archivos físicos de la tabla según la llave de clustering declarada. Sin este paso, la tabla existe y tiene la propiedad configurada, pero los archivos todavía no están reordenados; `FULL` fuerza una reorganización completa, a diferencia de un `OPTIMIZE` normal, más incremental.

3. `DESCRIBE TABLE EXTENDED` trae el detalle completo de la tabla, incluida la información de clustering; `SHOW TBLPROPERTIES` muestra puntualmente las propiedades internas de la tabla, entre ellas `clusteringColumns`, que confirma qué columnas quedaron configuradas como llave.

In [0]:
# Se debe aplicar el cluster by sobre un campo que se sabe que se usará mucho para consultas.

spark.sql("""
CREATE OR REPLACE TABLE dbassociate.default.sesion11_ventas_clustered
CLUSTER BY (tienda_id)
AS SELECT * FROM dbassociate.default.sesion11_ventas_hechos
""")

spark.sql("OPTIMIZE dbassociate.default.sesion11_ventas_clustered FULL")

# Nos muestra el detalle de la tabla y el clustering que tiene internamente.
display(spark.sql("DESCRIBE TABLE EXTENDED dbassociate.default.sesion11_ventas_clustered"))

# SHOW TBLPROPERTIES es la tabla con mayor detalle.
# Se puede aplicar el clustering en 4 columnas como máximo.
display(spark.sql("SHOW TBLPROPERTIES dbassociate.default.sesion11_ventas_clustered"))

Buscá `clusteringColumns` en la salida de `SHOW TBLPROPERTIES`: debería listar `tienda_id`. Si corrés `EXPLAIN` sobre un `SELECT * FROM dbassociate.default.sesion11_ventas_clustered WHERE tienda_id = 37` y lo comparás contra la misma consulta hecha sobre `sesion11_ventas_hechos` (sin clustering), la tabla clusterizada debería leer bastante menos archivos gracias al file skipping. `CLUSTER BY` no es compatible con particionamiento ni con Z-Ordering en la misma tabla: es uno o el otro.

## Lab 1K — Predictive optimization y `CLUSTER BY AUTO`

Predictive optimization corre por su cuenta el mantenimiento de tablas managed de Unity Catalog (compactación, vacuum, estadísticas), y es lo que sostiene la selección automática de llaves de clustering.

**Para explicarlo en clase:** es como tener a alguien de mantenimiento que reordena el archivador solo, sin que nadie se lo pida, fijándose en qué cajones abre más la gente. `CLUSTER BY AUTO` es pedirle a esa persona que, además, elija ella misma cuál es el mejor criterio de orden, mirando las consultas reales que se hicieron contra la tabla. Como recién armaste el archivador (la tabla), todavía no hay suficiente gente pidiendo papeles como para que decida un criterio: no es que la persona de mantenimiento esté fallando, es que todavía no tiene información para trabajar.

**Desglose del código, paso a paso:**

1. `DESCRIBE SCHEMA EXTENDED dbassociate.default` muestra la configuración del schema, incluido si predictive optimization está habilitado ahí (por herencia del catalog o del metastore, o por una configuración puntual de ese schema).

2. `ALTER TABLE ... CLUSTER BY AUTO` cambia la tabla que armaste en el Lab 1J de un `CLUSTER BY` con una llave fija (`tienda_id`, elegida a mano) a un modo automático, donde Databricks decide por su cuenta qué columnas conviene usar como llave.

3. El segundo `SHOW TBLPROPERTIES` es la misma consulta del Lab 1J, repetida después del cambio: sirve para comparar cómo se ve la propiedad de clustering antes (una llave fija, elegida por vos) y después (en modo automático, todavía sin ninguna llave asignada si la tabla es muy nueva).

In [0]:
display(spark.sql("DESCRIBE SCHEMA EXTENDED dbassociate.default"))

# decide de forma automatica la columna para realizar el clustering.
# el clustering se puede realizar con máximo 4 columnas.
# Cada insersión de datos se hacer el ordenamiento 
spark.sql("ALTER TABLE dbassociate.default.sesion11_ventas_clustered CLUSTER BY AUTO")

display(spark.sql("SHOW TBLPROPERTIES dbassociate.default.sesion11_ventas_clustered"))

En `DESCRIBE SCHEMA EXTENDED` fijate si predictive optimization aparece habilitado (heredado del catalog o del metastore, o configurado puntual en este schema). Con `CLUSTER BY AUTO`, Databricks elige las llaves de clustering solo, basándose en el historial real de consultas contra la tabla: sobre una tabla recién creada, en la que casi no hubo consultas todavía, **es esperable que la selección automática no elija ninguna llave por ahora**. No es un error del lab, es el comportamiento documentado cuando todavía no hay suficiente historial de consultas para decidir.

## Limpieza

In [0]:
dbutils.widgets.removeAll()

for tabla in [
    "dbassociate.default.sesion11_ventas_hechos",
    "dbassociate.default.sesion11_ventas_clustered",
]:
    spark.sql(f"DROP TABLE IF EXISTS {tabla}")

print("Tablas de este laboratorio eliminadas.")
print("Si vas a seguir directo con el Lab Reto, dejá el cluster sesion11_cluster_diagnostico corriendo.")
print("Si termina la sesión acá, terminalo desde Compute para no seguir consumiendo.")